# NLP II Lab Project
Multi-Class Text Classification

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, SimpleRNN, LSTM, GRU, Bidirectional

## Load Dataset

In [ ]:
df_train = pd.read_csv('Training_data_11.csv')
df_test = pd.read_csv('Test_data.csv')

In [ ]:
df_train.head()

## EDA

In [ ]:
df_train.info()

In [ ]:
df_train['label'].value_counts()

In [ ]:
sns.countplot(x='label', data=df_train)
plt.xticks(rotation=45)
plt.show()

## Preprocessing

In [ ]:
import re

def clean_text(text):
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z ]', '', text)
    text = text.lower()
    return text

In [ ]:
df_train['clean_text'] = df_train['text'].apply(clean_text)
df_test['clean_text'] = df_test['text'].apply(clean_text)

## TF-IDF

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train = vectorizer.fit_transform(df_train['clean_text'])
X_test = vectorizer.transform(df_test['clean_text'])

y_train = df_train['label']
y_test = df_test['label']

## Logistic Regression

In [ ]:
model_lr = LogisticRegression(max_iter=200)
model_lr.fit(X_train, y_train)

y_pred = model_lr.predict(X_test)

In [ ]:
print('Accuracy:', accuracy_score(y_test, y_pred))
print('F1 Score:', f1_score(y_test, y_pred, average='macro'))

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d')
plt.show()

## Deep Neural Network

In [ ]:
X_train_dense = X_train.toarray()
X_test_dense = X_test.toarray()

In [ ]:
model_dnn = Sequential()
model_dnn.add(Dense(128, activation='relu', input_shape=(X_train_dense.shape[1],)))
model_dnn.add(Dense(64, activation='relu'))
model_dnn.add(Dense(len(y_train.unique()), activation='softmax'))
model_dnn.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model_dnn.fit(X_train_dense, y_train, epochs=5, batch_size=32)

In [ ]:
loss, acc = model_dnn.evaluate(X_test_dense, y_test)
print('Accuracy:', acc)

## Tokenization for RNN

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(df_train['clean_text'])

X_train_seq = tokenizer.texts_to_sequences(df_train['clean_text'])
X_test_seq = tokenizer.texts_to_sequences(df_test['clean_text'])

X_train_pad = pad_sequences(X_train_seq, maxlen=100)
X_test_pad = pad_sequences(X_test_seq, maxlen=100)

## SimpleRNN

In [ ]:
model_rnn = Sequential()
model_rnn.add(Embedding(5000, 128, input_length=100))
model_rnn.add(SimpleRNN(64))
model_rnn.add(Dense(len(y_train.unique()), activation='softmax'))
model_rnn.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model_rnn.fit(X_train_pad, y_train, epochs=3)

## LSTM

In [ ]:
model_lstm = Sequential()
model_lstm.add(Embedding(5000, 128, input_length=100))
model_lstm.add(LSTM(64))
model_lstm.add(Dense(len(y_train.unique()), activation='softmax'))
model_lstm.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model_lstm.fit(X_train_pad, y_train, epochs=3)

## GRU

In [ ]:
model_gru = Sequential()
model_gru.add(Embedding(5000, 128, input_length=100))
model_gru.add(GRU(64))
model_gru.add(Dense(len(y_train.unique()), activation='softmax'))
model_gru.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model_gru.fit(X_train_pad, y_train, epochs=3)